# Notebook 03 - Autoencoder for Credit Card Fraud Detection

## Objective
To detect fraudulent transactions using an Autoencoder-based anomaly detection approach.

**Dataset:** Credit Card Fraud Detection (ULB)

**Learning Type:** Unsupervised Learning

**Author:** Sanchita Senapati


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

In [2]:
df = pd.read_csv("../data/processed/creditcard_clean.csv")

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (284807, 30)
Target Shape: (284807,)


In [4]:
X_normal = X[y == 0]

print("Normal transactions for training:", X_normal.shape)

Normal transactions for training: (284315, 30)


In [5]:
scaler = StandardScaler()

X_normal_scaled = scaler.fit_transform(X_normal)
X_scaled = scaler.transform(X)

print("Scaled normal data shape:", X_normal_scaled.shape)
print("Scaled complete data shape:", X_scaled.shape)

Scaled normal data shape: (284315, 30)
Scaled complete data shape: (284807, 30)


In [22]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

print("Scaler saved successfully!")

Scaler saved successfully!


In [6]:
input_dim = X_normal_scaled.shape[1]

input_layer = Input(shape=(input_dim,))

encoded = Dense(16, activation="relu")(input_layer)
encoded = Dense(8, activation="relu")(encoded)

decoded = Dense(16, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(input_layer, decoded)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,286 (5.02 KB)

 Trainable params: 1,286 (5.02 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = autoencoder.fit(
    X_normal_scaled,
    X_normal_scaled,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.7548 - val_loss: 0.6703
Epoch 2/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.5847 - val_loss: 0.6217
Epoch 3/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5424 - val_loss: 0.5858
Epoch 4/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.5097 - val_loss: 0.5623
Epoch 5/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.4921 - val_loss: 0.5478
Epoch 6/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.4842 - val_loss: 0.5405
Epoch 7/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.4784 - val_loss: 0.5390
Epoch 8/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4741 - val_loss: 0.5371
Epoch 9/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4704 - val_loss: 0.5354
Epoch 10/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4668 - val_loss: 0.5277


In [8]:
autoencoder.save("../models/autoencoder.keras")

print("Autoencoder saved successfully!")

Autoencoder saved successfully!


In [9]:
X_reconstructed = autoencoder.predict(X_scaled)

print("Original shape:", X_scaled.shape)
print("Reconstructed shape:", X_reconstructed.shape)

8901/8901 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step
Original shape: (284807, 30)
Reconstructed shape: (284807, 30)


In [10]:
reconstruction_error = np.mean(
    np.square(X_scaled - X_reconstructed),
    axis=1
)

print(reconstruction_error[:10])

[0.26610961 0.10447647 0.69161102 0.61345663 0.32498242 0.1454961
 0.28661712 0.48222324 0.13558967 0.23043258]


In [11]:
X_normal_reconstructed = autoencoder.predict(X_normal_scaled)

normal_reconstruction_error = np.mean(
    np.square(X_normal_scaled - X_normal_reconstructed),
    axis=1
)

print("Mean normal reconstruction error:",
      normal_reconstruction_error.mean())

print("Maximum normal reconstruction error:",
      normal_reconstruction_error.max())

8885/8885 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step
Mean normal reconstruction error: 0.4786878808005695
Maximum normal reconstruction error: 656.3285408328248


In [12]:
percentiles = np.percentile(
    normal_reconstruction_error,
    [90, 95, 99, 99.5, 99.9]
)

for p, value in zip([90, 95, 99, 99.5, 99.9], percentiles):
    print(f"{p}th percentile: {value:.4f}")

90th percentile: 0.8089
95th percentile: 1.0912
99th percentile: 2.5581
99.5th percentile: 4.3014
99.9th percentile: 12.9313


In [13]:
threshold = np.percentile(normal_reconstruction_error, 99)

autoencoder_predictions = (
    reconstruction_error > threshold
).astype(int)

print(autoencoder_predictions[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [14]:
print("Total anomalies detected:", autoencoder_predictions.sum())

Total anomalies detected: 3248


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

In [16]:
cm_autoencoder = confusion_matrix(y, autoencoder_predictions)

print(cm_autoencoder)

[[281471   2844]
 [    88    404]]


In [17]:
print(classification_report(y, autoencoder_predictions))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    284315
           1       0.12      0.82      0.22       492

    accuracy                           0.99    284807
   macro avg       0.56      0.91      0.61    284807
weighted avg       1.00      0.99      0.99    284807



In [18]:
print("Total predicted anomalies:", autoencoder_predictions.sum())
print("TP + FP:", cm_autoencoder[1, 1] + cm_autoencoder[0, 1])

Total predicted anomalies: 3248
TP + FP: 3248


In [19]:
TN, FP, FN, TP = cm_autoencoder.ravel()

false_positive_rate_autoencoder = FP / (FP + TN)

print("Autoencoder False Positive Rate:",
      false_positive_rate_autoencoder)

Autoencoder False Positive Rate: 0.010002989641770571


In [20]:
thresholds = {
    "95%": np.percentile(normal_reconstruction_error, 95),
    "99%": np.percentile(normal_reconstruction_error, 99),
    "99.5%": np.percentile(normal_reconstruction_error, 99.5),
    "99.9%": np.percentile(normal_reconstruction_error, 99.9)
}

for name, threshold in thresholds.items():
    predictions = (reconstruction_error > threshold).astype(int)
    
    cm = confusion_matrix(y, predictions)
    TN, FP, FN, TP = cm.ravel()
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = FP / (FP + TN)
    
    print(f"\nThreshold: {name} ({threshold:.4f})")
    print(f"TP: {TP}")
    print(f"FP: {FP}")
    print(f"FN: {FN}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"False Positive Rate: {fpr:.4f}")


Threshold: 95% (1.0912)
TP: 432
FP: 14216
FN: 60
Precision: 0.0295
Recall: 0.8780
F1-score: 0.0571
False Positive Rate: 0.0500

Threshold: 99% (2.5581)
TP: 404
FP: 2844
FN: 88
Precision: 0.1244
Recall: 0.8211
F1-score: 0.2160
False Positive Rate: 0.0100

Threshold: 99.5% (4.3014)
TP: 375
FP: 1422
FN: 117
Precision: 0.2087
Recall: 0.7622
F1-score: 0.3277
False Positive Rate: 0.0050

Threshold: 99.9% (12.9313)
TP: 205
FP: 285
FN: 287
Precision: 0.4184
Recall: 0.4167
F1-score: 0.4175
False Positive Rate: 0.0010


In [21]:
comparison = pd.DataFrame({
    "Metric": [
        "True Positives",
        "False Positives",
        "False Negatives",
        "Precision",
        "Recall",
        "F1-score",
        "False Positive Rate"
    ],
    
    "Isolation Forest": [
        126,
        366,
        366,
        0.26,
        0.26,
        0.26,
        0.00129
    ],
    
    "Autoencoder (99.5%)": [
        337,
        1422,
        155,
        0.1916,
        0.6850,
        0.2994,
        0.0050
    ]
})

comparison

,Metric,Isolation Forest,Autoencoder (99.5%)
0,True Positives,126.00000,337.0000
1,False Positives,366.00000,1422.0000
2,False Negatives,366.00000,155.0000
3,Precision,0.26000,0.1916
4,Recall,0.26000,0.6850
5,F1-score,0.26000,0.2994
6,False Positive Rate,0.00129,0.0050


In [23]:
import json

threshold = 4.7944

with open("../models/threshold.json", "w") as f:
    json.dump({"threshold": threshold}, f)

print("Threshold saved successfully!")

Threshold saved successfully!
